In [1]:
from ib_insync import *
import nest_asyncio
import pandas as pd
import numpy as numpy
import time

In [2]:
nest_asyncio.apply()

ib = IB()
#ib.connect('127.0.0.1', 7496, clientId=3)

Error 321, reqId -1: Error validating request.-'cp' : cause - The TWS/Gateway process is running in Read-Only mode. Trading actions are not allowed.
Error 321, reqId -1: Error validating request.-'bZ' : cause - The TWS/Gateway process is running in Read-Only mode. Trading actions are not allowed.
open orders request timed out
completed orders request timed out


<IB connected to 127.0.0.1:7496 clientId=3>

In [84]:
contract = Stock('NOW', 'SMART', 'USD')

bars = ib.reqHistoricalData(
    contract,
    endDateTime='',
    durationStr='6 M',
    barSizeSetting='5 mins',
    whatToShow='TRADES',
    useRTH=True
)

df = util.df(bars)
df['symbol'] = contract.symbol
df.set_index('date', inplace=True)


In [85]:
close_t1 = df.groupby(df.index.date)['close'].last()

high_t1 = df[df.index.time == pd.Timestamp('10:00').time()].set_index(df[df.index.time == pd.Timestamp('10:00').time()].index.date)['high']


result = pd.DataFrame({
    'close_t2': close_t1.shift(1),
    'high_t1': high_t1,
    'close_t1': close_t1,
    'first_30min_spike': (high_t1/close_t1.shift(1)) - 1,
    'daily_return': ((close_t1/high_t1 ) - 1)*100.0
})
result.tail()

,close_t2,high_t1,close_t1,first_30min_spike,daily_return
2026-05-22,99.66,103.25,102.11,0.036022,-1.104116
2026-05-26,102.11,101.56,100.04,-0.005386,-1.496652
2026-05-27,100.04,102.29,102.17,0.022491,-0.117314
2026-05-28,102.17,109.75,108.75,0.074190,-0.911162
2026-05-29,108.75,121.66,124.39,0.118713,2.243959


In [86]:
trades = result[result['first_30min_spike'] > 0.05]
trades['daily_return'].describe()

count    9.000000
mean    -0.425682
std      2.742720
min     -7.191687
25%     -0.518701
50%     -0.281620
75%      0.877792
max      2.243959
Name: daily_return, dtype: float64

In [82]:
print(f"Winning notches: {len(trades[trades['daily_return'] > 0])}")
print(f"Total Trade: {len(trades)}")

Winning notches: 4
Total Trade: 7


In [ ]:
df['10ma']=df['close'].ewm(span=10).mean()
df['20ma']=df['close'].ewm(span=20).mean()
df['50ma']=df['close'].ewm(span=50).mean()
df['trend_acceleration'] = ((df['10ma'] - df['50ma']) / df['close']).ewm(span=20).mean()
df['10ma_greater_20ma'] = df['10ma'] > df['20ma']
df.head(30)

## Data

### Getting all tickers 

In [ ]:
# https://www.nasdaq.com/market-activity/stocks/screener?page=1&rows_per_page=25



### Now Fetching Historical Data of these Tickers from IBKR

In [23]:
ticker_list_df = pd.read_csv('nasdaqlisted.txt', sep='|')
tickers = ticker_list_df['Symbol'].tolist()[:-1] 

In [7]:
def chunk_list(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

batches = list(chunk_list(tickers, 50))  # 50 per batch

In [9]:
def get_daily_bars(ticker, n_days=25):
    contract = Stock(ticker, 'SMART', 'USD')
    bars = ib.reqHistoricalData(
        contract,
        endDateTime='',
        durationStr='2 M',        # ~2 months covers 22+ trading days
        barSizeSetting='1 day',
        whatToShow='TRADES',
        useRTH=True,
        formatDate=1
    )
    return util.df(bars).tail(n_days)

def compute_metrics(df):
    closes = df['close']
    highs  = df['high']
    lows   = df['low']
    volume = df['volume']

    # close / 22-day low rank (your primary sort key)
    low_22  = lows.tail(22).min()
    close_22_rank = closes.iloc[-1] / low_22

    # ADR% over last 20 days
    adr_pct = ((highs.tail(20) - lows.tail(20)) / lows.tail(20)).mean() * 100

    # Dollar volume (last close × last volume)
    dollar_vol = closes.iloc[-1] * volume.iloc[-1]

    return {
        'close_low_rank': close_22_rank,
        'adr_pct': adr_pct,
        'dollar_vol_usd': dollar_vol
    }

In [ ]:
# --- Main loop ---

results = []
for tkr in tickers:
    try:
        df = get_daily_bars(tkr)
        m  = compute_metrics(df)
        m['ticker'] = tkr
        results.append(m)
        ib.sleep(0.5)   # be gentle on the API
    except Exception as e:
        print(f"{tkr}: {e}")

df_out = pd.DataFrame(results)

# Apply filters
df_filtered = df_out[
    (df_out['adr_pct']      >= 2.5) &
    (df_out['dollar_vol_usd'] >= 60e6)
].sort_values('close_low_rank').reset_index(drop=True)

print(df_filtered[['ticker','close_low_rank','adr_pct','dollar_vol_usd']].to_string())

In [19]:
import pandas as pd
from pathlib import Path
import logging
from ib_insync import IB, Stock, util

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger(__name__)

DATA_DIR = Path('./data/daily')
DATA_DIR.mkdir(parents=True, exist_ok=True)

DURATION  = '6 M'
BAR_SIZE  = '1 day'

DEAD_TICKERS_FILE = Path('./data/dead_tickers.txt')

def load_dead_tickers():
    if DEAD_TICKERS_FILE.exists():
        return set(DEAD_TICKERS_FILE.read_text().splitlines())
    return set()

def save_dead_tickers(dead: set):
    DEAD_TICKERS_FILE.write_text('\n'.join(sorted(dead)))

def fetch_ticker(ib: IB, ticker: str, force_refresh: bool = False):
    out_path = DATA_DIR / f'{ticker}.parquet'

    if out_path.exists() and not force_refresh:
        log.info(f'{ticker}: using cached data')
        return pd.read_parquet(out_path)

    contract = Stock(ticker, 'SMART', 'USD')

    try:
        ib.qualifyContracts(contract)
    except Exception as e:
        log.warning(f'{ticker}: qualify failed — {e}')
        return 'dead'

    try:
        bars = ib.reqHistoricalData(
            contract,
            endDateTime='',
            durationStr=DURATION,
            barSizeSetting=BAR_SIZE,
            whatToShow='TRADES',
            useRTH=True,
            formatDate=1,
            timeout=30,
        )
    except Exception as e:
        log.warning(f'{ticker}: data request failed — {e}')
        return None

    if not bars or len(bars) < 20:
        log.warning(f'{ticker}: insufficient data ({len(bars) if bars else 0} bars)')
        return 'dead'

    df = util.df(bars)[['date', 'open', 'high', 'low', 'close', 'volume']]
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date').sort_index()
    df.to_parquet(out_path)
    log.info(f'{ticker}: fetched {len(df)} bars → saved')
    return df

def fetch_all(tickers: list[str], force_refresh: bool = False, sleep: float = 0.5):
    ib = IB()
    ib.connect('127.0.0.1', 7496, clientId=5)

    dead = load_dead_tickers()
    skipped, fetched, failed = 0, 0, 0

    for i, tkr in enumerate(tickers):
        if tkr in dead:
            skipped += 1
            continue

        result = fetch_ticker(ib, tkr, force_refresh=force_refresh)

        if result is None:
            pass
        elif isinstance(result, str) and result == 'dead':
            dead.add(tkr)
            failed += 1
            save_dead_tickers(dead)
        else:
            fetched += 1

        ib.sleep(sleep)

        if (i + 1) % 50 == 0:
            log.info(f'Progress: {i+1}/{len(tickers)} | fetched={fetched} failed={failed} skipped={skipped}')

    ib.disconnect()
    log.info(f'Final: fetched={fetched} failed={failed} skipped={skipped}')
    log.info(f'Dead tickers ({len(dead)}): {sorted(dead)}')
    return dead

In [20]:
fetch_all(tickers)

2026-04-18 22:27:07,802 INFO Connecting to 127.0.0.1:7496 with clientId 5...
2026-04-18 22:27:07,804 INFO Connected
2026-04-18 22:27:07,808 INFO Logged on to server version 176
2026-04-18 22:27:07,815 INFO Warning 2104, reqId -1: Market data farm connection is OK:hfarm
2026-04-18 22:27:07,816 INFO Warning 2104, reqId -1: Market data farm connection is OK:eufarmnj
2026-04-18 22:27:07,817 INFO Warning 2104, reqId -1: Market data farm connection is OK:usfuture
2026-04-18 22:27:07,817 INFO Warning 2104, reqId -1: Market data farm connection is OK:afarm
2026-04-18 22:27:07,817 INFO Warning 2104, reqId -1: Market data farm connection is OK:usopt.nj
2026-04-18 22:27:07,818 INFO Warning 2104, reqId -1: Market data farm connection is OK:usfarm.nj
2026-04-18 22:27:07,818 INFO Warning 2104, reqId -1: Market data farm connection is OK:eufarm
2026-04-18 22:27:07,818 INFO Warning 2104, reqId -1: Market data farm connection is OK:usfarm
2026-04-18 22:27:07,819 INFO Warning 2106, reqId -1: HMDS data f

ConnectionError: Socket disconnect

In [27]:
import os

completed_tickers = [Path(f).stem for f in DATA_DIR.iterdir() if f.suffix == '.parquet']
failed  = load_dead_tickers()

retry = set(tickers) - set(completed_tickers) - failed
print(len(retry))

2080


In [28]:
len(set(failed))

276

In [29]:
failed

{'AACI',
 'AACIW',
 'AACO',
 'AACOW',
 'AACPU',
 'ABLVW',
 'ABVEW',
 'ACAA',
 'ACAAW',
 'ACGCU',
 'ACONW',
 'ADACW',
 'ADSEW',
 'AEAQW',
 'AENTW',
 'AERTW',
 'AFRIW',
 'AIIOW',
 'AIMDW',
 'AIRJW',
 'AISPW',
 'ALCYW',
 'ALDFW',
 'ALFUW',
 'ALMR',
 'ALOVW',
 'ALVOW',
 'AMODW',
 'AMPGR',
 'AMPGZ',
 'ANABV',
 'ANGHW',
 'ANNAW',
 'ANSCW',
 'APLMW',
 'APXTW',
 'ARBEW',
 'ARCIW',
 'AREBW',
 'ARQQW',
 'ARTCW',
 'ARXS',
 'ASBPW',
 'ASND',
 'ASPSW',
 'ASPSZ',
 'ASTLW',
 'ATIIW',
 'AUROW',
 'BAERW',
 'BBCQW',
 'BBLGW',
 'BCARW',
 'BCGWW',
 'BCTXL',
 'BCTXZ',
 'BDCIW',
 'BDMDW',
 'BEATW',
 'BENFW',
 'BETRW',
 'BFRGW',
 'BFRIW',
 'BGLWW',
 'BHAV',
 'BHAVR',
 'BIAFW',
 'BIVIW',
 'BIXIW',
 'BLRKW',
 'BLUWW',
 'BLZRW',
 'BNAIW',
 'BNCWW',
 'BNCWZ',
 'BNZIW',
 'BRLSW',
 'BRRWW',
 'BTBDW',
 'BTMWW',
 'BULLW',
 'BZAIW',
 'BZFDW',
 'CAQ',
 'CAQUW',
 'CCGWW',
 'CCIIW',
 'CCIXW',
 'CCXIW',
 'CDIOW',
 'CDROW',
 'CDTTW',
 'CELUW',
 'CGCTW',
 'CHECW',
 'CINGW',
 'CLSKW',
 'CMCT',
 'CMIIW',
 'CNCKW',
 'COCHW',


In [35]:
cadl_df.head()

,open,high,low,close,volume
date,,,,,
2025-10-20,5.37,5.6430,5.3600,5.54,517066.0
2025-10-21,5.65,5.7000,5.3807,5.49,338277.0
2025-10-22,5.42,5.4900,5.1100,5.26,734794.0
2025-10-23,5.25,5.5200,5.2300,5.31,433171.0
2025-10-24,5.39,5.5051,5.3300,5.40,460271.0


In [46]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('./data/daily')

def load_ticker(ticker):
    p = DATA_DIR / f'{ticker}.parquet'
    if not p.exists():
        return None
    return pd.read_parquet(p)

def compute_row(ticker):
    df = load_ticker(ticker)
    if df is None or len(df) < 124:
        return None

    close = df['close']
    high  = df['high']
    low   = df['low']

    # ADR% filter — last 20 days
    adr = ((high.tail(20) - low.tail(20)) / low.tail(20)).mean() * 100
    if adr < 2.5:
        return None

    last = close.iloc[-1]

    # lookbacks in trading days
    ret_1m  = last / close.iloc[-22]  - 1
    ret_3m  = last / close.iloc[-63]  - 1
    ret_6m  = last / close.iloc[-124] - 1

    volume = df["volume"]
    avg_dollarvol = (close.tail(20)*volume.tail(20)).mean()

    return {
        'ticker': ticker,
        'adr_pct': round(adr, 2),
        'avg_dollarvol_usd': round(avg_dollarvol, 2),
        'ret_1m':  round(ret_1m  * 100, 2),
        'ret_3m':  round(ret_3m  * 100, 2),
        'ret_6m':  round(ret_6m  * 100, 2),
    }

def run_screen():
    tickers = [f.stem for f in DATA_DIR.iterdir() if f.suffix == '.parquet']

    rows = [compute_row(t) for t in tickers]
    df   = pd.DataFrame([r for r in rows if r is not None]).set_index('ticker')

    # rank each lookback — higher return = better rank (percentile 0-100)
    for col in ['ret_1m', 'ret_3m', 'ret_6m']:
        df[f'{col}_rank'] = df[col].rank(pct=True) * 100

    # composite score — equal weight, tweak as you like
    df['momentum_score'] = (df['ret_1m_rank'] + df['ret_3m_rank'] + df['ret_6m_rank']) / 3

    # filter out low volume and low adr
    df = df[(df['adr_pct'] >= 2.5) & (df['avg_dollarvol_usd'] >= 60e6)]

    # top 7%
    threshold = df['momentum_score'].quantile(0.90)
    top       = df[df['momentum_score'] >= threshold].sort_values('momentum_score', ascending=False)

    print(top[['adr_pct', 'ret_1m', 'ret_3m', 'ret_6m', 'momentum_score']].to_string())
    return top

In [47]:
result_df = run_screen()

        adr_pct  ret_1m  ret_3m   ret_6m  momentum_score
ticker                                                  
CAR       16.96  386.47  293.11   231.21       99.429895
AEHR      12.19  130.96  191.08   191.18       98.913237
AAOI      13.90   72.10  330.40   366.96       98.735079
AXTI      18.66   69.32  273.74  1581.47       98.735079
FORM       6.61   42.66   75.21   210.78       95.938001
BIRD      27.83  245.05  166.01    52.11       95.902369
ALM        9.92   31.72  170.13   160.22       94.975949
LITE       9.07   27.58  175.73   455.32       94.815607
MRVL       5.25   59.43   73.61    62.73       94.388028
LGN        6.45   43.16   48.76   107.10       93.746660
CNTA       3.21   38.14   78.65    74.02       93.443791
INTC       5.81   52.12   45.87    79.79       93.283449
AMKR       6.70   43.74   40.35   108.71       93.140923
FLY       13.01   88.53   30.86    61.21       92.232318
LUNR      12.29   52.29   27.80   113.96       92.089792
GSAT       6.59   36.37   33.01

In [45]:
len(result_df)

12